# 🎵 Análisis de Música y Salud Mental - Minería de Datos

**Integrantes:** Alejandra Guzman - Macarena Lobos  
**Sección:** 002D  
**Dataset:** Music and Mental Health Survey  
**Objetivo:** Analizar la relación entre hábitos de consumo de música y problemas de salud mental

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print('✅ Librerías importadas')

In [ ]:
# Cargar datos
df = pd.read_csv('musica_y_salud_mental.csv')
print(f'Datos cargados: {df.shape}')
df.head()

## 📊 EXPLORACIÓN Y LIMPIEZA

In [ ]:
# Limpiar
df_clean = df.copy()
df_clean.columns = df_clean.columns.str.replace('&amp;', '&')
df_clean = df_clean.replace('R&amp;B', 'R&B')

# Faltantes
df_clean['Age'].fillna(df_clean['Age'].median(), inplace=True)
df_clean['BPM'].fillna(df_clean['BPM'].median(), inplace=True)
df_clean['Music effects'].fillna(df_clean['Music effects'].mode()[0], inplace=True)
df_clean = df_clean.drop(['Timestamp', 'Permissions'], axis=1)

print(f'Dataset limpio: {df_clean.shape}')
print(f'Faltantes: {df_clean.isnull().sum().sum()}')

## 🧠 PREPARACIÓN PARA MACHINE LEARNING

In [ ]:
df_ml = df_clean.copy()

# Codificar
binary_cols = ['While working', 'Instrumentalist', 'Composer', 'Exploratory', 'Foreign languages']
for col in binary_cols:
    df_ml[col] = (df_ml[col] == 'Yes').astype(int)

le_service = LabelEncoder()
df_ml['Primary streaming service'] = le_service.fit_transform(df_ml['Primary streaming service'])

le_genre = LabelEncoder()
df_ml['Fav genre'] = le_genre.fit_transform(df_ml['Fav genre'])

le_effects = LabelEncoder()
df_ml['Music effects'] = le_effects.fit_transform(df_ml['Music effects'])

freq_cols = [col for col in df_ml.columns if 'Frequency' in col]
freq_map = {'Never': 0, 'Rarely': 1, 'Sometimes': 2, 'Very frequently': 3}
for col in freq_cols:
    df_ml[col] = df_ml[col].map(freq_map)

print('✅ Datos codificados')

## 🎯 K-MEANS: SEGMENTACIÓN

In [ ]:
# Características
features_km = ['Age', 'Hours per day', 'BPM', 'Anxiety', 'Depression', 'Insomnia', 'OCD']
X_km = df_ml[features_km].copy()

# Normalizar
scaler = StandardScaler()
X_km_scaled = scaler.fit_transform(X_km)

# Encontrar k óptimo
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_temp.fit(X_km_scaled)
    silhouettes.append(silhouette_score(X_km_scaled, km_temp.labels_))

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'k óptimo: {best_k}')

In [ ]:
# Entrenar K-Means
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_ml['Cluster'] = kmeans.fit_predict(X_km_scaled)

print(f'Clusters: {df_ml["Cluster"].value_counts().sort_index().to_dict()}')
print(f'Silhouette: {silhouette_score(X_km_scaled, kmeans.labels_):.3f}')

In [ ]:
# Visualizar con PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_km_scaled)

plt.figure(figsize=(10, 7))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for cluster in range(best_k):
    mask = df_ml['Cluster'] == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {cluster}', 
               alpha=0.6, s=100, color=colors[cluster % len(colors)])

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('K-Means Clustering')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 🌳 ÁRBOL DE DECISIÓN

In [ ]:
# Preparar datos
X_tree = df_ml.drop(['Music effects', 'Cluster'], axis=1)
y_tree = df_ml['Music effects']

X_train, X_test, y_train, y_test = train_test_split(
    X_tree, y_tree, test_size=0.2, random_state=42, stratify=y_tree
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Encontrar profundidad óptima
max_depths = range(1, 16)
train_accs = []
test_accs = []

for depth in max_depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(dt.score(X_train, y_train))
    test_accs.append(dt.score(X_test, y_test))

best_depth = max_depths[np.argmax(test_accs)]
print(f'Profundidad óptima: {best_depth}')

In [ ]:
# Entrenar modelo final
dt_final = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
dt_final.fit(X_train, y_train)

y_train_pred = dt_final.predict(X_train)
y_test_pred = dt_final.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f'Train Accuracy: {train_acc:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Overfitting: {train_acc - test_acc:.4f}')

In [ ]:
# Reporte
print(classification_report(y_test, y_test_pred, target_names=['Improve', 'No effect', 'Worsen']))

In [ ]:
# Importancia de características
feat_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': dt_final.feature_importances_
}).sort_values('Importance', ascending=False)

print(feat_imp.head(10))